<a href="https://colab.research.google.com/github/HeshanNavindu-7/oil-spill-detection/blob/main/Edge_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
# ============================================================
# EO1: Edge Optimization Path Setup
# ============================================================

import os
import time
import shutil
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

# ------------------------------------------------------------
# Your model folders
# ------------------------------------------------------------
BASELINE_PROJECT_DIR = Path("/content/drive/MyDrive/Oil Spill/Baseline Detection Model")
BEST_PROJECT_DIR = Path("/content/drive/MyDrive/Oil Spill/Best Detection Model")

# Edge optimization should use the BEST model folder
PROJECT_DIR = BEST_PROJECT_DIR

MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"
TFLITE_DIR = PROJECT_DIR / "tflite"
PLOT_DIR = PROJECT_DIR / "plots"
LOG_DIR = PROJECT_DIR / "logs"

for folder in [MODEL_DIR, RESULT_DIR, TFLITE_DIR, PLOT_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Dataset Image Dimensions
# ------------------------------------------------------------
IMG_WIDTH = 192
IMG_HEIGHT = 192
IMG_CHANNELS = 3

# ------------------------------------------------------------
# Model paths
# ------------------------------------------------------------
BASELINE_MODEL_PATH = BASELINE_PROJECT_DIR / "models" / "baseline_unet_model.keras"

BEST_MODEL_PATH = BEST_PROJECT_DIR / "models" / "best_edge_unet_model.keras"

# Fallback if best_edge_unet_model.keras was not saved but stage2 exists
STAGE2_MODEL_PATH = BEST_PROJECT_DIR / "models" / "edge_unet_stage2_best.keras"

if not BEST_MODEL_PATH.exists() and STAGE2_MODEL_PATH.exists():
    print("best_edge_unet_model.keras not found. Using stage2 best model instead.")
    BEST_MODEL_PATH = STAGE2_MODEL_PATH

# ------------------------------------------------------------
# Edge output paths
# ------------------------------------------------------------
SAVED_MODEL_DIR = TFLITE_DIR / "saved_model_edge_unet"

FP16_TFLITE_PATH = TFLITE_DIR / "edge_unet_fp16.tflite"
INT8_TFLITE_PATH = TFLITE_DIR / "edge_unet_int8.tflite"

# ------------------------------------------------------------
# Dataset paths (assuming a 'dataset' folder with 'train/images' and 'test/images')
# ------------------------------------------------------------
DATA_DIR = PROJECT_DIR / "dataset"
TRAIN_IMAGES_DIR = DATA_DIR / "train" / "images"
TEST_IMAGES_DIR = DATA_DIR / "test" / "images"
PROCESSED_DIR = PROJECT_DIR / "processed_dataset" # Added PROCESSED_DIR

# ------------------------------------------------------------
# Utility functions for dataset loading (reused for INT8 and test inference)
# ------------------------------------------------------------
def load_image(image_path):
    image = tf.io.read_file(image_path)
    # Assuming JPEG. Adjust if PNG/other. Use tf.image.decode_image for auto-detection.
    image = tf.image.decode_jpeg(image, channels=IMG_CHANNELS)
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    image = tf.cast(image, tf.float32) / 255.0 # Normalize to [0, 1]
    return image

def create_dummy_ds(batch_size=1, num_samples=100, image_size=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)):
    """Creates a dummy tf.data.Dataset yielding (image, dummy_label) tuples."""
    def generator():
        for _ in range(num_samples):
            yield tf.random.uniform(shape=image_size, dtype=tf.float32), tf.constant(0, dtype=tf.float32)
    output_signature = (
        tf.TensorSpec(shape=image_size, dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.float32) # Dummy label for compatibility
    )
    ds = tf.data.Dataset.from_generator(generator, output_signature=output_signature)
    return ds.batch(batch_size)

print("Baseline project:", BASELINE_PROJECT_DIR)
print("Baseline model exists:", BASELINE_MODEL_PATH.exists())

print("Best project:", BEST_PROJECT_DIR)
print("Best model path:", BEST_MODEL_PATH)
print("Best model exists:", BEST_MODEL_PATH.exists())

print("TFLite output folder:", TFLITE_DIR)
print("\n--- Dataset Paths ---")
print("Train images folder:", TRAIN_IMAGES_DIR, ", exists:", TRAIN_IMAGES_DIR.exists())
print("Test images folder:", TEST_IMAGES_DIR, ", exists:", TEST_IMAGES_DIR.exists())
print("Processed dataset folder:", PROCESSED_DIR, ", exists:", PROCESSED_DIR.exists())

Baseline project: /content/drive/MyDrive/Oil Spill/Baseline Detection Model
Baseline model exists: True
Best project: /content/drive/MyDrive/Oil Spill/Best Detection Model
Best model path: /content/drive/MyDrive/Oil Spill/Best Detection Model/models/best_edge_unet_model.keras
Best model exists: True
TFLite output folder: /content/drive/MyDrive/Oil Spill/Best Detection Model/tflite

--- Dataset Paths ---
Train images folder: /content/drive/MyDrive/Oil Spill/Best Detection Model/dataset/train/images , exists: False
Test images folder: /content/drive/MyDrive/Oil Spill/Best Detection Model/dataset/test/images , exists: False
Processed dataset folder: /content/drive/MyDrive/Oil Spill/Best Detection Model/processed_dataset , exists: False


In [26]:
# ============================================================
# EO2 Updated: Correct Folder Listing
# ============================================================

from pathlib import Path
import os

BEST_PROJECT_DIR = Path("/content/drive/MyDrive/Oil Spill/Best Detection Model")
BASELINE_PROJECT_DIR = Path("/content/drive/MyDrive/Oil Spill/Baseline Detection Model")

PROJECT_DIR = BEST_PROJECT_DIR

LOG_DIR = PROJECT_DIR / "logs"
MODEL_DIR = PROJECT_DIR / "models"
PLOT_DIR = PROJECT_DIR / "plots"
PROCESSED_DIR = PROJECT_DIR / "processed"   # IMPORTANT: processed, not processed_dataset
RESULT_DIR = PROJECT_DIR / "results"
TFLITE_DIR = PROJECT_DIR / "tflite"

BASELINE_MODEL_PATH = BASELINE_PROJECT_DIR / "models" / "baseline_unet_model.keras"
BEST_MODEL_PATH = MODEL_DIR / "best_edge_unet_model.keras"

FP16_TFLITE_PATH = TFLITE_DIR / "edge_unet_fp16.tflite"
INT8_TFLITE_PATH = TFLITE_DIR / "edge_unet_int8.tflite"

def list_folder(folder_path, title):
    print(f"\n=== {title} ===")
    folder_path = Path(folder_path)

    if not folder_path.exists():
        print("Folder not found:", folder_path)
        return

    files = sorted(folder_path.iterdir())

    if len(files) == 0:
        print("Empty folder")
        return

    for f in files:
        if f.is_dir():
            print("[DIR] ", f.name)
        else:
            print("[FILE]", f.name)

list_folder(MODEL_DIR, "models folder")
list_folder(LOG_DIR, "logs folder")
list_folder(RESULT_DIR, "results folder")
list_folder(TFLITE_DIR, "tflite folder")

for split in ["train", "valid", "test"]:
    list_folder(PROCESSED_DIR / split / "images", f"processed/{split}/images")
    list_folder(PROCESSED_DIR / split / "masks", f"processed/{split}/masks")

Streaming output truncated to the last 5000 lines.
[FILE] R-4-_jpg.rf.7eda28c64abde7db4f654e1bddfa9467.jpg
[FILE] R-40-_jpg.rf.16bc12e27999ddc7bcf2a81fba8e46d4.jpg
[FILE] R-41-_jpg.rf.0bc77e5cf7c51327de6b89a3a5402fb0.jpg
[FILE] R-41-_jpg.rf.450b165aa062a4c500749bbbacd04293.jpg
[FILE] R-42-_jpg.rf.4d475cb3a8eed288cf8a7642b76c11c4.jpg
[FILE] R-42-_jpg.rf.618540ef42a344ac0d12b2e937ae9913.jpg
[FILE] R-45-_jpg.rf.0eedd2b450807c97b5ea8aab1adc7bd9.jpg
[FILE] R-45-_jpg.rf.2d229dc1b46aa9113654acb88bb6714e.jpg
[FILE] R-46-_jpg.rf.76249f7980b81af48f69d0b06fae9978.jpg
[FILE] R-46-_jpg.rf.923848510d63f6c9427ff5b484000b8f.jpg
[FILE] R-46-_jpg.rf.f8dc749bafe31707b4084e32baa6eede.jpg
[FILE] R-47-_jpg.rf.2e52afe726dbc9772eaef865038cc03d.jpg
[FILE] R-49-_jpg.rf.096eedd8ba694ac294cde160d0732a62.jpg
[FILE] R-5-_jpg.rf.2cf8af3a8fb3ce0e4238f7501445f8a0.jpg
[FILE] R-5-_jpg.rf.47eba6fc215877a553dc6e345623678a.jpg
[FILE] R-50-_jpg.rf.d394b196d712fc28247f09872c01c4be.jpg
[FILE] R-52-_jpg.rf.c8577cc3afd4ef77fccb

In [27]:
# ============================================================
# EO3: Rebuild Test Dataset for Latency Benchmark
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd
import time

IMG_SIZE = 192
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE

def get_image_mask_paths(project_path: Path, split_name: str):
    img_dir = project_path / "processed" / split_name / "images"
    mask_dir = project_path / "processed" / split_name / "masks"

    image_map = {
        Path(f).stem: str(img_dir / f)
        for f in os.listdir(img_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    }

    mask_map = {
        Path(f).stem: str(mask_dir / f)
        for f in os.listdir(mask_dir)
        if f.lower().endswith(".png")
    }

    paired_keys = sorted(set(image_map.keys()) & set(mask_map.keys()))

    image_paths = [image_map[k] for k in paired_keys]
    mask_paths = [mask_map[k] for k in paired_keys]

    return image_paths, mask_paths

def read_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def read_mask(path):
    mask = tf.io.read_file(path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")
    mask = tf.cast(mask > 127, tf.float32)
    return mask

def load_pair(image_path, mask_path):
    return read_image(image_path), read_mask(mask_path)

def make_dataset(image_paths, mask_paths):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    ds = ds.map(load_pair, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

test_image_paths, test_mask_paths = get_image_mask_paths(PROJECT_DIR, "test")
test_ds = make_dataset(test_image_paths, test_mask_paths)

print("Test images:", len(test_image_paths))
print("Test masks:", len(test_mask_paths))

for images, _ in test_ds.take(1):
    sample_input = images[:1].numpy()
    print("Sample input shape:", sample_input.shape)
    print("Sample min/max:", sample_input.min(), sample_input.max())

Test images: 343
Test masks: 343
Sample input shape: (1, 192, 192, 3)
Sample min/max: -1.0 1.0


In [28]:
# ============================================================
# EO3: Rebuild Test Dataset for Latency Benchmark
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd
import time

IMG_SIZE = 192
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE

def get_image_mask_paths(project_path: Path, split_name: str):
    img_dir = project_path / "processed" / split_name / "images"
    mask_dir = project_path / "processed" / split_name / "masks"

    image_map = {
        Path(f).stem: str(img_dir / f)
        for f in os.listdir(img_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    }

    mask_map = {
        Path(f).stem: str(mask_dir / f)
        for f in os.listdir(mask_dir)
        if f.lower().endswith(".png")
    }

    paired_keys = sorted(set(image_map.keys()) & set(mask_map.keys()))

    image_paths = [image_map[k] for k in paired_keys]
    mask_paths = [mask_map[k] for k in paired_keys]

    return image_paths, mask_paths

def read_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def read_mask(path):
    mask = tf.io.read_file(path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")
    mask = tf.cast(mask > 127, tf.float32)
    return mask

def load_pair(image_path, mask_path):
    return read_image(image_path), read_mask(mask_path)

def make_dataset(image_paths, mask_paths):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    ds = ds.map(load_pair, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

test_image_paths, test_mask_paths = get_image_mask_paths(PROJECT_DIR, "test")
test_ds = make_dataset(test_image_paths, test_mask_paths)

print("Test images:", len(test_image_paths))
print("Test masks:", len(test_mask_paths))

for images, _ in test_ds.take(1):
    sample_input = images[:1].numpy()
    print("Sample input shape:", sample_input.shape)
    print("Sample min/max:", sample_input.min(), sample_input.max())

Test images: 343
Test masks: 343
Sample input shape: (1, 192, 192, 3)
Sample min/max: -1.0 1.0


In [29]:
# ============================================================
# EO10: Proper TFLite Latency Benchmark
# ============================================================

import time
import numpy as np
import pandas as pd
from pathlib import Path

def file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return None
    return path.stat().st_size / (1024 * 1024)


def benchmark_tflite(model_path, sample_input, warmup=10, runs=100):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    input_dtype = input_details[0]["dtype"]
    input_scale, input_zero_point = input_details[0]["quantization"]

    x = sample_input.copy()

    if input_dtype == np.int8:
        x = np.round(x / input_scale + input_zero_point).astype(np.int8)
    else:
        x = x.astype(input_dtype)

    # Warm-up runs
    for _ in range(warmup):
        interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()

    times = []

    for _ in range(runs):
        start = time.perf_counter()

        interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()

        end = time.perf_counter()
        times.append((end - start) * 1000)

    return {
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "p95_ms": float(np.percentile(times, 95)),
        "min_ms": float(np.min(times)),
        "max_ms": float(np.max(times)),
        "runs": runs,
        "warmup": warmup
    }


# Get one sample input from test set
for images, _ in test_ds.take(1):
    sample_input = images[:1].numpy()
    break

fp16_benchmark = benchmark_tflite(FP16_TFLITE_PATH, sample_input)
int8_benchmark = benchmark_tflite(INT8_TFLITE_PATH, sample_input)

latency_df = pd.DataFrame([
    {
        "model_format": "TFLite FP16",
        "size_mb": file_size_mb(FP16_TFLITE_PATH),
        **fp16_benchmark
    },
    {
        "model_format": "TFLite INT8",
        "size_mb": file_size_mb(INT8_TFLITE_PATH),
        **int8_benchmark
    }
])

latency_path = RESULT_DIR / "edge_optimization_tflite_latency_benchmark.csv"
latency_df.to_csv(latency_path, index=False)

print("Saved latency benchmark:", latency_path)
latency_df

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Saved latency benchmark: /content/drive/MyDrive/Oil Spill/Best Detection Model/results/edge_optimization_tflite_latency_benchmark.csv


,model_format,size_mb,mean_ms,std_ms,p95_ms,min_ms,max_ms,runs,warmup
0,TFLite FP16,2.381962,40.426247,24.887885,102.462903,15.469239,120.339939,100,10
1,TFLite INT8,1.584251,17.735446,15.037886,49.600801,13.532428,112.455029,100,10


In [30]:
# ============================================================
# EO11: Full Test-Set Evaluation for TFLite FP16 and INT8
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from pathlib import Path

# If best_threshold already exists from threshold tuning, this will use it.
# If not, load it from threshold sweep CSV.
try:
    print("Using existing best_threshold:", best_threshold)
except NameError:
    threshold_df = pd.read_csv(RESULT_DIR / "edge_unet_threshold_sweep_results.csv")
    best_row = threshold_df.loc[threshold_df["dice"].idxmax()]
    best_threshold = float(best_row["threshold"])
    print("Loaded best_threshold from CSV:", best_threshold)


def calculate_metrics_from_predictions(y_true, y_prob, threshold):
    y_true_bin = (y_true > 0.5).astype(np.uint8).reshape(-1)
    y_pred_bin = (y_prob > threshold).astype(np.uint8).reshape(-1)

    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    def safe_div(a, b):
        return float(a / b) if b else 0.0

    return {
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": safe_div(tp + tn, tn + fp + fn + tp),
        "precision": safe_div(tp, tp + fp),
        "recall": safe_div(tp, tp + fn),
        "specificity": safe_div(tn, tn + fp),
        "iou": safe_div(tp, tp + fp + fn),
        "dice": safe_div(2 * tp, 2 * tp + fp + fn),
    }


def run_tflite_batch(model_path, input_batch):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype = input_details[0]["dtype"]
    output_dtype = output_details[0]["dtype"]

    input_scale, input_zero_point = input_details[0]["quantization"]
    output_scale, output_zero_point = output_details[0]["quantization"]

    outputs = []

    for i in range(input_batch.shape[0]):
        x = input_batch[i:i+1].copy()

        if input_dtype == np.int8:
            x = np.round(x / input_scale + input_zero_point).astype(np.int8)
        else:
            x = x.astype(input_dtype)

        interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()

        y = interpreter.get_tensor(output_details[0]["index"])

        if output_dtype == np.int8:
            y = (y.astype(np.float32) - output_zero_point) * output_scale
        else:
            y = y.astype(np.float32)

        y = np.clip(y, 0.0, 1.0)
        outputs.append(y)

    return np.concatenate(outputs, axis=0)


def collect_tflite_predictions(model_path, dataset):
    all_true = []
    all_prob = []

    for images, masks in dataset:
        probs = run_tflite_batch(model_path, images.numpy())

        all_true.append(masks.numpy())
        all_prob.append(probs)

    y_true = np.concatenate(all_true, axis=0)
    y_prob = np.concatenate(all_prob, axis=0)

    return y_true, y_prob


# Keras predictions
def collect_keras_predictions(model, dataset):
    all_true = []
    all_prob = []

    for images, masks in dataset:
        probs = model.predict(images, verbose=0)

        all_true.append(masks.numpy())
        all_prob.append(probs)

    y_true = np.concatenate(all_true, axis=0)
    y_prob = np.concatenate(all_prob, axis=0)

    return y_true, y_prob


# Load Keras best model
best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={
        "focal_dice_loss": focal_dice_loss,
        "dice_coef": dice_coef
    }
)

# Evaluate all formats
evaluation_rows = []

print("Evaluating Keras model...")
y_true_keras, y_prob_keras = collect_keras_predictions(best_model, test_ds)
keras_metrics = calculate_metrics_from_predictions(y_true_keras, y_prob_keras, best_threshold)
keras_metrics["model_format"] = "Keras FP32"
keras_metrics["size_mb"] = file_size_mb(BEST_MODEL_PATH)
evaluation_rows.append(keras_metrics)

print("Evaluating TFLite FP16 model...")
y_true_fp16, y_prob_fp16 = collect_tflite_predictions(FP16_TFLITE_PATH, test_ds)
fp16_metrics = calculate_metrics_from_predictions(y_true_fp16, y_prob_fp16, best_threshold)
fp16_metrics["model_format"] = "TFLite FP16"
fp16_metrics["size_mb"] = file_size_mb(FP16_TFLITE_PATH)
evaluation_rows.append(fp16_metrics)

print("Evaluating TFLite INT8 model...")
y_true_int8, y_prob_int8 = collect_tflite_predictions(INT8_TFLITE_PATH, test_ds)
int8_metrics = calculate_metrics_from_predictions(y_true_int8, y_prob_int8, best_threshold)
int8_metrics["model_format"] = "TFLite INT8"
int8_metrics["size_mb"] = file_size_mb(INT8_TFLITE_PATH)
evaluation_rows.append(int8_metrics)

eval_df = pd.DataFrame(evaluation_rows)

# Reorder columns
cols = [
    "model_format", "size_mb", "threshold",
    "accuracy", "precision", "recall", "specificity", "iou", "dice",
    "tn", "fp", "fn", "tp"
]
eval_df = eval_df[cols]

eval_path = RESULT_DIR / "edge_optimization_keras_vs_tflite_accuracy.csv"
eval_df.to_csv(eval_path, index=False)

print("Saved:", eval_path)
eval_df

Loaded best_threshold from CSV: 0.6200000000000003
Evaluating Keras model...
Evaluating TFLite FP16 model...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluating TFLite INT8 model...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Saved: /content/drive/MyDrive/Oil Spill/Best Detection Model/results/edge_optimization_keras_vs_tflite_accuracy.csv


,model_format,size_mb,threshold,accuracy,precision,recall,specificity,iou,dice,tn,fp,fn,tp
0,Keras FP32,12.049788,0.62,0.901982,0.796662,0.828318,0.927309,0.683759,0.812181,8725264,683963,555414,2679711
1,TFLite FP16,2.381962,0.62,0.902009,0.796426,0.828875,0.927154,0.683965,0.812327,8723807,685420,553611,2681514
2,TFLite INT8,1.584251,0.62,0.744145,0.000000,0.000000,1.000000,0.000000,0.000000,9409227,0,3235125,0


In [31]:
# ============================================================
# EO12: INT8 Output Diagnostic Check
# ============================================================

def collect_tflite_output_stats(model_path, dataset, max_batches=5):
    all_outputs = []

    for batch_idx, (images, masks) in enumerate(dataset):
        if batch_idx >= max_batches:
            break

        probs = run_tflite_batch(model_path, images.numpy())
        all_outputs.append(probs)

    outputs = np.concatenate(all_outputs, axis=0)

    print("Model:", Path(model_path).name)
    print("Output shape:", outputs.shape)
    print("Output min:", float(outputs.min()))
    print("Output max:", float(outputs.max()))
    print("Output mean:", float(outputs.mean()))
    print("Output std:", float(outputs.std()))
    print("Percentiles:")
    for p in [50, 75, 90, 95, 99, 99.5, 99.9]:
        print(f"P{p}:", float(np.percentile(outputs, p)))

    return outputs


print("FP16 output stats")
fp16_outputs = collect_tflite_output_stats(FP16_TFLITE_PATH, test_ds)

print("\nINT8 output stats")
int8_outputs = collect_tflite_output_stats(INT8_TFLITE_PATH, test_ds)

FP16 output stats


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Model: edge_unet_fp16.tflite
Output shape: (40, 192, 192, 1)
Output min: 0.00033765853731893003
Output max: 0.9999350905418396
Output mean: 0.37221014499664307
Output std: 0.3820214569568634
Percentiles:
P50: 0.16829988360404968
P75: 0.7794543504714966
P90: 0.9854377508163452
P95: 0.9959031939506531
P99: 0.9994545578956604
P99.5: 0.9996473789215088
P99.9: 0.9998308420181274

INT8 output stats
Model: edge_unet_int8.tflite
Output shape: (40, 192, 192, 1)
Output min: 0.0
Output max: 0.5
Output mean: 0.06686719506978989
Output std: 0.10410251468420029
Percentiles:
P50: 0.02734375
P75: 0.05859375
P90: 0.16015625
P95: 0.31640625
P99: 0.5
P99.5: 0.5
P99.9: 0.5


In [32]:
# ============================================================
# EO13: INT8 Separate Threshold Sweep
# ============================================================

# Collect full INT8 predictions again
y_true_int8, y_prob_int8 = collect_tflite_predictions(INT8_TFLITE_PATH, test_ds)

# INT8 output max is around 0.5, so test lower thresholds carefully
int8_thresholds = np.concatenate([
    np.arange(0.01, 0.10, 0.01),
    np.arange(0.10, 0.31, 0.02),
    np.arange(0.30, 0.51, 0.02)
])

int8_threshold_results = []

for th in int8_thresholds:
    row = calculate_metrics_from_predictions(y_true_int8, y_prob_int8, th)
    row["threshold"] = float(th)
    int8_threshold_results.append(row)

int8_threshold_df = pd.DataFrame(int8_threshold_results)

# Sort by Dice score
int8_threshold_df = int8_threshold_df.sort_values(
    by=["dice", "iou", "accuracy"],
    ascending=False
)

int8_threshold_path = RESULT_DIR / "int8_separate_threshold_sweep_results.csv"
int8_threshold_df.to_csv(int8_threshold_path, index=False)

print("Saved:", int8_threshold_path)

print("Top 10 INT8 thresholds:")
int8_threshold_df.head(10)

Saved: /content/drive/MyDrive/Oil Spill/Best Detection Model/results/int8_separate_threshold_sweep_results.csv
Top 10 INT8 thresholds:


,threshold,tn,fp,fn,tp,accuracy,precision,recall,specificity,iou,dice
3,0.04,6943032,2466195,1373699,1861426,0.696315,0.430127,0.575380,0.737896,0.326490,0.492262
4,0.05,7379032,2030195,1531069,1704056,0.718351,0.456331,0.526736,0.784234,0.323638,0.489013
2,0.03,5748553,3660674,1025051,2210074,0.629421,0.376455,0.683149,0.610948,0.320496,0.485417
5,0.06,7816384,1592843,1715508,1519617,0.738353,0.488237,0.469724,0.830715,0.314753,0.478802
6,0.07,8013791,1395436,1807400,1427725,0.746698,0.505719,0.441320,0.851695,0.308327,0.471330
1,0.02,4150912,5258315,664962,2570163,0.531548,0.328309,0.794456,0.441153,0.302606,0.464616
7,0.08,8217264,1191963,1911057,1324068,0.754592,0.526253,0.409279,0.873320,0.299083,0.460453
8,0.09,8389228,1019999,2010447,1224678,0.760332,0.545592,0.378557,0.891596,0.287813,0.446979
9,0.10,8488394,920833,2074821,1160304,0.763084,0.557534,0.358658,0.902135,0.279191,0.436511
0,0.01,627395,8781832,75762,3159363,0.299482,0.264577,0.976581,0.066679,0.262909,0.416354


In [34]:
# ============================================================
# EO14: Final Edge Optimization Decision Summary
# ============================================================

import json
import pandas as pd
from pathlib import Path

# Load existing result files
accuracy_path = RESULT_DIR / "edge_optimization_keras_vs_tflite_accuracy.csv"
latency_path = RESULT_DIR / "edge_optimization_tflite_latency_benchmark.csv"
int8_sweep_path = RESULT_DIR / "int8_separate_threshold_sweep_results.csv"

accuracy_df = pd.read_csv(accuracy_path)
latency_df = pd.read_csv(latency_path)
int8_sweep_df = pd.read_csv(int8_sweep_path)

# Best INT8 after separate threshold tuning
best_int8_row = int8_sweep_df.sort_values(by="dice", ascending=False).iloc[0]

# Extract FP16 and Keras rows
keras_row = accuracy_df[accuracy_df["model_format"] == "Keras FP32"].iloc[0]
fp16_row = accuracy_df[accuracy_df["model_format"] == "TFLite FP16"].iloc[0]

fp16_latency = latency_df[latency_df["model_format"] == "TFLite FP16"].iloc[0]
int8_latency = latency_df[latency_df["model_format"] == "TFLite INT8"].iloc[0]

final_edge_summary = {
    "final_selected_model": "TFLite FP16",
    "reason": "FP16 preserved Keras-level accuracy while significantly reducing model size. INT8 was smaller and faster but caused a large accuracy drop.",
    "keras_fp32": {
        "size_mb": float(keras_row["size_mb"]),
        "threshold": float(keras_row["threshold"]),
        "accuracy": float(keras_row["accuracy"]),
        "precision": float(keras_row["precision"]),
        "recall": float(keras_row["recall"]),
        "iou": float(keras_row["iou"]),
        "dice": float(keras_row["dice"])
    },
    "tflite_fp16": {
        "size_mb": float(fp16_row["size_mb"]),
        "threshold": float(fp16_row["threshold"]),
        "accuracy": float(fp16_row["accuracy"]),
        "precision": float(fp16_row["precision"]),
        "recall": float(fp16_row["recall"]),
        "iou": float(fp16_row["iou"]),
        "dice": float(fp16_row["dice"]),
        "mean_latency_ms": float(fp16_latency["mean_ms"]),
        "p95_latency_ms": float(fp16_latency["p95_ms"])
    },
    "tflite_int8_best_threshold": {
        "size_mb": float(int8_latency["size_mb"]),
        "threshold": float(best_int8_row["threshold"]),
        "accuracy": float(best_int8_row["accuracy"]),
        "precision": float(best_int8_row["precision"]),
        "recall": float(best_int8_row["recall"]),
        "iou": float(best_int8_row["iou"]),
        "dice": float(best_int8_row["dice"]),
        "mean_latency_ms": float(int8_latency["mean_ms"]),
        "p95_latency_ms": float(int8_latency["p95_ms"])
    }
}

# Save JSON
json_path = RESULT_DIR / "final_edge_optimization_decision_summary.json"
with open(json_path, "w") as f:
    json.dump(final_edge_summary, f, indent=2)

# Save comparison CSV
final_table = pd.DataFrame([
    {
        "model": "Keras FP32",
        "size_mb": keras_row["size_mb"],
        "threshold": keras_row["threshold"],
        "accuracy": keras_row["accuracy"],
        "precision": keras_row["precision"],
        "recall": keras_row["recall"],
        "iou": keras_row["iou"],
        "dice": keras_row["dice"],
        "mean_latency_ms": None,
        "p95_latency_ms": None,
        "decision": "Original trained model"
    },
    {
        "model": "TFLite FP16",
        "size_mb": fp16_row["size_mb"],
        "threshold": fp16_row["threshold"],
        "accuracy": fp16_row["accuracy"],
        "precision": fp16_row["precision"],
        "recall": fp16_row["recall"],
        "iou": fp16_row["iou"],
        "dice": fp16_row["dice"],
        "mean_latency_ms": fp16_latency["mean_ms"],
        "p95_latency_ms": fp16_latency["p95_ms"],
        "decision": "Selected final edge model"
    },
    {
        "model": "TFLite INT8",
        "size_mb": int8_latency["size_mb"],
        "threshold": best_int8_row["threshold"],
        "accuracy": best_int8_row["accuracy"],
        "precision": best_int8_row["precision"],
        "recall": best_int8_row["recall"],
        "iou": best_int8_row["iou"],
        "dice": best_int8_row["dice"],
        "mean_latency_ms": int8_latency["mean_ms"],
        "p95_latency_ms": int8_latency["p95_ms"],
        "decision": "Rejected due to accuracy degradation"
    }
])

csv_path = RESULT_DIR / "final_edge_optimization_comparison_table.csv"
final_table.to_csv(csv_path, index=False)

# Save Markdown summary
md_text = f"""
# Final Edge Optimization Decision Summary

## Final Selected Model
The final selected edge-optimized detection model is **TFLite FP16**.

## Reason for Selection
The original Keras FP32 model achieved a Dice score of {keras_row['dice']:.4f} and IoU of {keras_row['iou']:.4f} with a size of {keras_row['size_mb']:.2f} MB.
After FP16 TensorFlow Lite conversion, the model size reduced to {fp16_row['size_mb']:.2f} MB while maintaining almost identical performance with Dice {fp16_row['dice']:.4f} and IoU {fp16_row['iou']:.4f}.

Although INT8 quantization reduced the model size further to {int8_latency['size_mb']:.2f} MB and achieved faster average latency, its best threshold-tuned Dice score was only {best_int8_row['dice']:.4f}. Therefore, INT8 was not selected as the final deployment model.

## Final Comparison

| Model | Size MB | Threshold | Accuracy | Precision | Recall | IoU | Dice | Mean Latency ms | Decision |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---|
| Keras FP32 | {keras_row['size_mb']:.2f} | {keras_row['threshold']:.2f} | {keras_row['accuracy']:.4f} | {keras_row['precision']:.4f} | {keras_row['recall']:.4f} | {keras_row['iou']:.4f} | {keras_row['dice']:.4f} | - | Original model |
| TFLite FP16 | {fp16_row['size_mb']:.2f} | {fp16_row['threshold']:.2f} | {fp16_row['accuracy']:.4f} | {fp16_row['precision']:.4f} | {fp16_row['recall']:.4f} | {fp16_row['iou']:.4f} | {fp16_row['dice']:.4f} | {fp16_latency['mean_ms']:.2f} | Selected |
| TFLite INT8 | {int8_latency['size_mb']:.2f} | {best_int8_row['threshold']:.2f} | {best_int8_row['accuracy']:.4f} | {best_int8_row['precision']:.4f} | {best_int8_row['recall']:.4f} | {best_int8_row['iou']:.4f} | {best_int8_row['dice']:.4f} | {int8_latency['mean_ms']:.2f} | Rejected |

## Final Statement
The FP16 TensorFlow Lite model was selected as the final edge-optimized oil spill detection model because it achieved a strong balance between model size reduction and detection accuracy preservation.
"""

md_path = RESULT_DIR / "FINAL_EDGE_OPTIMIZATION_DECISION_SUMMARY.md"
md_path.write_text(md_text)

print("Saved JSON:", json_path)
print("Saved CSV:", csv_path)
print("Saved Markdown:", md_path)

final_table

Saved JSON: /content/drive/MyDrive/Oil Spill/Best Detection Model/results/final_edge_optimization_decision_summary.json
Saved CSV: /content/drive/MyDrive/Oil Spill/Best Detection Model/results/final_edge_optimization_comparison_table.csv
Saved Markdown: /content/drive/MyDrive/Oil Spill/Best Detection Model/results/FINAL_EDGE_OPTIMIZATION_DECISION_SUMMARY.md


,model,size_mb,threshold,accuracy,precision,recall,iou,dice,mean_latency_ms,p95_latency_ms,decision
0,Keras FP32,12.049788,0.62,0.901982,0.796662,0.828318,0.683759,0.812181,NaN,NaN,Original trained model
1,TFLite FP16,2.381962,0.62,0.902009,0.796426,0.828875,0.683965,0.812327,40.426247,102.462903,Selected final edge model
2,TFLite INT8,1.584251,0.04,0.696315,0.430127,0.575380,0.326490,0.492262,17.735446,49.600801,Rejected due to accuracy degradation
